In [2]:
import pandas as pd
import numpy as np
import time

In [3]:
np.random.seed(42)
n = 100_000

df_raw = pd.DataFrame({
    "user_id":  np.arange(1, n + 1),
    "age":      np.random.randint(18, 80, size=n).astype(float),
    "score":    np.random.normal(loc=60, scale=15, size=n),
    "income":   np.random.normal(loc=50000, scale=15000, size=n),
    "category": np.random.choice(["A", "B", "C", None], size=n),
})

# inject missing values
missing_idx = np.random.choice(n, size=5000, replace=False)
df_raw.loc[missing_idx[:2000], "age"] = np.nan
df_raw.loc[missing_idx[2000:4000], "score"] = np.nan
df_raw.loc[missing_idx[4000:], "income"] = np.nan

# inject outliers
outlier_idx = np.random.choice(n, size=500, replace=False)
df_raw.loc[outlier_idx[:250], "score"] = np.random.choice([200, -100], size=250)
df_raw.loc[outlier_idx[250:], "income"] = np.random.choice([500000, -99999], size=250)

print("Raw dataset shape:", df_raw.shape)
df_raw.isnull().sum()

Raw dataset shape: (100000, 5)


user_id         0
age          2000
score        1993
income        997
category    24875
dtype: int64

In [4]:
sample = df_raw["score"].copy()

# LOOP (slow)
start = time.time()
result_loop = []

for val in sample:
    if pd.isna(val):
        result_loop.append(sample.mean())
    else:
        result_loop.append(val)

loop_time = time.time() - start

# VECTORIZED (fast)
start = time.time()
result_vec = sample.fillna(sample.mean())
vec_time = time.time() - start

print("=" * 50)
print("BENCHMARK RESULTS")
print("Loop time:", loop_time)
print("Vectorized time:", vec_time)
print("Speedup:", loop_time / vec_time)
print("=" * 50)

BENCHMARK RESULTS
Loop time: 1.5394136905670166
Vectorized time: 0.0017104148864746094
Speedup: 900.0235572902146


In [5]:
df = df_raw.copy()

numeric_cols = ["age", "score", "income"]
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

df["category"] = df["category"].fillna(df["category"].mode()[0])

df.isnull().sum()

user_id     0
age         0
score       0
income      0
category    0
dtype: int64

In [6]:
def remove_outliers_iqr(dataframe, cols):
    mask = pd.Series([True] * len(dataframe), index=dataframe.index)

    for col in cols:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        mask &= dataframe[col].between(lower, upper)

    return dataframe[mask]

In [7]:
def remove_outliers_zscore(dataframe, cols, threshold=3):
    mask = pd.Series([True] * len(dataframe), index=dataframe.index)

    for col in cols:
        mean = dataframe[col].mean()
        std = dataframe[col].std()

        z = (dataframe[col] - mean) / std
        mask &= z.abs() <= threshold

    return dataframe[mask]

In [8]:
cols_to_check = ["score", "income"]

df_iqr = remove_outliers_iqr(df, cols_to_check)
df_zscore = remove_outliers_zscore(df, cols_to_check)

print("Original rows:", len(df))
print("IQR cleaned:", len(df_iqr))
print("Z-score cleaned:", len(df_zscore))

Original rows: 100000
IQR cleaned: 97873
Z-score cleaned: 99403


In [9]:
print("""
WHY RESULTS DIFFER:

IQR method:
- Based on quartiles
- Works well with skewed data
- More robust to extreme values

Z-score method:
- Based on mean and standard deviation
- Assumes normal distribution
- Sensitive to outliers
""")


WHY RESULTS DIFFER:

IQR method:
- Based on quartiles
- Works well with skewed data
- More robust to extreme values

Z-score method:
- Based on mean and standard deviation
- Assumes normal distribution
- Sensitive to outliers



In [10]:
df_clean = df_iqr.copy()

df_raw.head(100).to_csv("raw_sample.csv", index=False)
df_clean.head(100).to_csv("clean_sample.csv", index=False)

print("Files saved successfully!")
print("Clean dataset shape:", df_clean.shape)

Files saved successfully!
Clean dataset shape: (97873, 5)
